### **Data Cleaning**

**Cleaning the fuel price data**

In [1]:
import pandas as pd
import numpy as np

fuel_raw = pd.read_csv('/Users/prishasureka/changi-fuel-risk/data/raw/eia_jet_fuel_weekly.csv', skiprows=3)

print(fuel_raw.head())
print(fuel_raw.dtypes)

   Apr 06, 1990  0.548
0  Apr 13, 1990  0.540
1  Apr 20, 1990  0.543
2  Apr 27, 1990  0.530
3  May 04, 1990  0.522
4  May 11, 1990  0.519
Apr 06, 1990     object
0.548           float64
dtype: object


Renaming column names, dropping rows where date parsing failed or price is missing, filtering to our study period (2005-2026), and sorting chronologically.

In [2]:
fuel_raw.columns = ['date', 'jet_fuel_usd_per_gallon']

fuel_raw['date'] = pd.to_datetime(fuel_raw['date'], errors='coerce')

fuel_raw = fuel_raw.dropna(subset=['date', 'jet_fuel_usd_per_gallon'])

fuel_raw = fuel_raw[fuel_raw['date'] >= '2005-01-01']
fuel_raw = fuel_raw[fuel_raw['date'] <= '2026-05-31']

fuel_raw = fuel_raw.sort_values('date').reset_index(drop=True)

print(f"Rows: {len(fuel_raw)}")
print(f"Date range: {fuel_raw['date'].min()} to {fuel_raw['date'].max()}")
print(f"Price range: ${fuel_raw['jet_fuel_usd_per_gallon'].min():.2f} - ${fuel_raw['jet_fuel_usd_per_gallon'].max():.2f}")
print(fuel_raw.isnull().sum())

Rows: 1116
Date range: 2005-01-07 00:00:00 to 2026-05-22 00:00:00
Price range: $0.46 - $4.63
date                       0
jet_fuel_usd_per_gallon    0
dtype: int64


Converting to monthly averages, because our arrivals data is monthly and we need them on the same time scale.


In [3]:
fuel_raw['year_month'] = fuel_raw['date'].dt.to_period('M')

fuel_monthly = (fuel_raw
    .groupby('year_month')['jet_fuel_usd_per_gallon']
    .mean()
    .reset_index()
)

fuel_monthly['year_month'] = fuel_monthly['year_month'].astype(str)

print(fuel_monthly.head(10))
print(f"Monthly rows: {len(fuel_monthly)}")

fuel_monthly.to_csv('../data/processed/fuel_monthly.csv', index=False)
print("Saved fuel_monthly.csv")

  year_month  jet_fuel_usd_per_gallon
0    2005-01                  1.33550
1    2005-02                  1.33375
2    2005-03                  1.55225
3    2005-04                  1.57740
4    2005-05                  1.46900
5    2005-06                  1.64750
6    2005-07                  1.66340
7    2005-08                  1.82775
8    2005-09                  2.21900
9    2005-10                  2.42325
Monthly rows: 257
Saved fuel_monthly.csv


**Cleaning the Singapore Arrivals Data**

In [5]:
arrivals_raw = pd.read_csv('../data/raw/sg_arrivals_by_country.csv')
print(arrivals_raw.head())
print(arrivals_raw.shape)
print(arrivals_raw.columns.tolist())

                         DataSeries  2025Jul  2025Jun  2025May  2025Apr  \
0  Number Of Air Passenger Arrivals  3042743  2962918  2873379  2923118   
1                   South East Asia  1173824  1174308  1158057  1130801   
2                           Vietnam   114128   113251   104090   106027   
3                         Indonesia   360848   359994   352346   321654   
4                     Other Regions    14828    11628    10863    11602   

   2025Mar  2025Feb  2025Jan  2024Dec  2024Nov  ...  1961Oct  1961Sep  \
0  2766949  2754111  3130805  3218469  2839182  ...    13619    12974   
1  1110012  1120833  1189806  1244205  1118406  ...       na       na   
2   105443    99305   101487   105848    98116  ...       na       na   
3   291880   295695   319040   341960   296314  ...     1953     1808   
4     9531     9166    11104    12588     9179  ...       na       na   

   1961Aug  1961Jul  1961Jun  1961May  1961Apr  1961Mar  1961Feb  1961Jan  
0    13501    13655    11532    12

The data.gov.sg arrivals dataset is typically in wide format — each country is a column and each row is a month. We will reshape it into long format (one row per country-month).

In [8]:
print(arrivals_raw.columns[:5])

arrivals_raw = arrivals_raw.rename(columns={arrivals_raw.columns[0]: 'year_month'})

arrivals_raw['year_month'] = pd.to_datetime(arrivals_raw['year_month'], 
                                              format='%Y %b',
                                              errors='coerce').dt.to_period('M').astype(str)

arrivals_raw = arrivals_raw.dropna(subset=['year_month'])

arrivals_long = arrivals_raw.melt(
    id_vars=['year_month'],
    var_name='country',
    value_name='arrivals'
)

arrivals_long['country'] = arrivals_long['country'].str.strip()

arrivals_long['arrivals'] = (
    arrivals_long['arrivals']
    .astype(str)
    .str.replace(',', '', regex=False)
)

arrivals_long['arrivals'] = pd.to_numeric(
    arrivals_long['arrivals'],
    errors='coerce'
)

arrivals_long = arrivals_long.dropna(subset=['arrivals'])

arrivals_long = arrivals_long[arrivals_long['arrivals'] > 0]

arrivals_long['arrivals'] = arrivals_long['arrivals'].astype(int)

arrivals_long = arrivals_long[arrivals_long['year_month'] >= '2005-01']
arrivals_long = arrivals_long[arrivals_long['year_month'] <= '2026-05']

print(arrivals_long.head(10))
print(f"Rows: {len(arrivals_long)}")
print(f"Countries: {arrivals_long['country'].nunique()}")
print(f"Date range: {arrivals_long['year_month'].min()} to {arrivals_long['year_month'].max()}")

arrivals_long.to_csv('../data/processed/arrivals_long.csv', index=False)
print("Saved arrivals_long.csv")

Index(['year_month', '2025Jul', '2025Jun', '2025May', '2025Apr'], dtype='object')
Empty DataFrame
Columns: [year_month, country, arrivals]
Index: []
Rows: 0
Countries: 0
Date range: nan to nan
Saved arrivals_long.csv


**Cleaning the World Bank GDP Data**

In [9]:
gdp_raw = pd.read_csv('../data/raw/world_bank_gdp_raw.csv')
print(gdp_raw.head())
print(gdp_raw.columns.tolist())
print(gdp_raw.dtypes)

                       country iso3  year  gdp_per_capita
0  Africa Eastern and Southern  AFE  2024     1615.396356
1  Africa Eastern and Southern  AFE  2023     1571.449189
2  Africa Eastern and Southern  AFE  2022     1679.327622
3  Africa Eastern and Southern  AFE  2021     1562.416175
4  Africa Eastern and Southern  AFE  2020     1351.591669
['country', 'iso3', 'year', 'gdp_per_capita']
country            object
iso3               object
year                int64
gdp_per_capita    float64
dtype: object


The wbdata output has a multi-index of (country, date). After reset_index() it has columns for country name, country code, date, and value.

In [11]:
gdp_raw.columns = [c.lower().replace(' ', '_') for c in gdp_raw.columns]

print("Columns:", gdp_raw.columns.tolist())

gdp_clean = gdp_raw[['country', 'iso3', 'year', 'gdp_per_capita']].copy()

gdp_clean = gdp_clean.dropna(subset=['year', 'gdp_per_capita'])

gdp_clean['year'] = gdp_clean['year'].astype(int)

gdp_clean = gdp_clean[
    (gdp_clean['year'] >= 2005) &
    (gdp_clean['year'] <= 2024)
]

print(f"GDP rows: {len(gdp_clean)}")
print(f"Countries: {gdp_clean['country'].nunique()}")
print(f"Year range: {gdp_clean['year'].min()} to {gdp_clean['year'].max()}")

gdp_clean.to_csv('../data/processed/gdp_clean.csv', index=False)
print("Saved gdp_clean.csv")

Columns: ['country', 'iso3', 'year', 'gdp_per_capita']
GDP rows: 5157
Countries: 262
Year range: 2005 to 2024
Saved gdp_clean.csv


**Cleaning the airline classification file**

In [16]:
airlines = pd.read_csv('../data/raw/airline_classifications.csv')
print(airlines.head())
print(airlines['carrier_type'].value_counts())
print(f"Classified: {len(airlines)} airlines")
print(f"Missing classifications: {airlines['carrier_type'].isnull().sum()}")

airlines['carrier_type'] = airlines['carrier_type'].str.upper().str.strip()
airlines['carrier_type'] = airlines['carrier_type'].replace({
    'LOW COST': 'LCC', 'LOW-COST': 'LCC',
    'FULL SERVICE': 'FSC', 'FULL-SERVICE': 'FSC'
})

airlines['carrier_type'] = airlines['carrier_type'].fillna('Unknown')

airlines.to_csv('../data/processed/airline_classifications_clean.csv', index=False)
print("Saved airline_classifications_clean.csv")

  airline_iata carrier_type
0           SQ          FSC
1           TR          LCC
2           3K          LCC
3           EK          FSC
4           QR          FSC
carrier_type
FSC    57
LCC    20
Name: count, dtype: int64
Classified: 77 airlines
Missing classifications: 0
Saved airline_classifications_clean.csv
